In [2]:
import cv2
import torch
from ultralytics import YOLO

# ---------------- CONFIG ----------------
USE_WEBCAM = True        # True = webcam, False = video
VIDEO_PATH = "t.mp4"

MODEL_PATHS = [
    "models/dms.pt",
    "models/dms2.pt"
]

CONFIDENCE = 0.5

# ---------------- SAFE MODEL LOADER ----------------
def load_model():
    for path in MODEL_PATHS:
        try:
            print(f"Trying to load: {path}")
            model = YOLO(path)
            print(f"✅ Loaded model: {path}")
            print("Classes:", model.names)
            return model
        except Exception as e:
            print(f"❌ Failed: {path} -> {e}")

    print("\n⚠️ Falling back to yolov8n.pt")
    model = YOLO("yolov8n.pt")
    print("Classes:", model.names)
    return model


# ---------------- MAIN ----------------
def main():
    model = load_model()
    CLASS_NAMES = model.names

    # Video source
    cap = cv2.VideoCapture(0 if USE_WEBCAM else VIDEO_PATH)

    if not cap.isOpened():
        print("❌ Error opening video source")
        return

    print("\n🚀 Running... Press 'q' to exit\n")

    while True:
        ret, frame = cap.read()

        if not ret:
            if not USE_WEBCAM:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue
            break

        # -------- INFERENCE --------
        results = model(frame, conf=CONFIDENCE, verbose=False)

        alert = "NORMAL"

        # -------- PROCESS DETECTIONS --------
        for box in results[0].boxes:
            cls = int(box.cls[0])
            label = CLASS_NAMES[cls]

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, label, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

            # -------- DMS LOGIC --------
            label_lower = label.lower()

            if any(k in label_lower for k in ["drowsy", "sleep", "eye", "closed"]):
                alert = "🚨 DROWSINESS ALERT"

            elif any(k in label_lower for k in ["phone", "mobile"]):
                alert = "📱 PHONE DETECTED"

            elif "yawn" in label_lower:
                alert = "😴 YAWNING"

        # -------- ALERT UI --------
        if alert != "NORMAL":
            overlay = frame.copy()
            cv2.rectangle(overlay, (0, 0), (frame.shape[1], 90), (0, 0, 255), -1)
            cv2.addWeighted(overlay, 0.35, frame, 0.65, 0, frame)

            cv2.putText(frame, alert,
                        (30, 60),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1.2,
                        (255,255,255),
                        3)

        # Status box
        cv2.rectangle(frame, (10, 100), (350, 160), (0,0,0), -1)
        cv2.putText(frame, f"STATUS: {alert}",
                    (20, 140),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0,255,255),
                    2)

        # -------- DISPLAY --------
        cv2.imshow("DMS System", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # -------- CLEANUP --------
    cap.release()
    cv2.destroyAllWindows()


# ---------------- ENTRY ----------------
if __name__ == "__main__":
    main()

Trying to load: models/dms.pt
❌ Failed: models/dms.pt -> 'model'
Trying to load: models/dms2.pt
❌ Failed: models/dms2.pt -> Can't get attribute 'DFLoss' on <module 'ultralytics.utils.loss' from 'C:\\Users\\soham\\anaconda3\\envs\\FinalPipeline\\lib\\site-packages\\ultralytics\\utils\\loss.py'>

⚠️ Falling back to yolov8n.pt


100%|█████████████████████████████████████████████████████████████████████████████| 6.23M/6.23M [00:01<00:00, 5.60MB/s]


Classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68